In [ ]:
import json
import pathlib
import itertools
import subprocess

import xarray as xr
from ecmwf.opendata import Client

## Download from ECMWF

In [36]:
client = Client(source="ecmwf")
client

In [38]:
result = client.retrieve(
    type="fc",
    time=0,
    step=list(range(0, 144, 3)),
    date="2026-04-24",
    param=["10u", "10v", "2t", "tp"],
    target="2026-04-24-surface.grib2",
)

vars(result)

Recovering from HTTP error [429 Too Many Requests], attempt 1 of 500
Retrying in 120 seconds


KeyboardInterrupt: 

## Post-Processing GRIB

In [39]:
root_path = pathlib.Path("./2026-04-24")
root_path.mkdir(parents=True, exist_ok=True)
root_path

PosixPath('2026-04-24')

In [40]:
result = subprocess.run(
    ["gdalinfo", "-json", "2026-04-24.grib2"], capture_output=True, text=True
)
data = json.loads(result.stdout)

result.returncode

0

In [41]:
all_bands = list([x["metadata"][""]["GRIB_ELEMENT"] for x in data["bands"]])
set(all_bands)

{'SOILTMP', 'TMP', 'UGRD', 'VGRD', 'VSOILM', 'unknown'}

In [82]:
BAND_NAMES = {
    "TMP": "2t",
    "UGRD": "10u",
    "VGRD": "10v",
    "unknown": "tp",
}

bands = {"2t": [], "tp": [], "10u": [], "10v": []}

timestamps = {"2t": [], "tp": [], "10u": [], "10v": []}

for band in data["bands"]:
    band_num = band["band"]
    element = band["metadata"][""]["GRIB_ELEMENT"]
    ref_time = int(band["metadata"][""]["GRIB_REF_TIME"])
    forecast_secs = int(band["metadata"][""]["GRIB_FORECAST_SECONDS"])

    if element not in BAND_NAMES.keys():
        continue

    bands[BAND_NAMES[element]].append(band_num)
    timestamps[BAND_NAMES[element]].append(ref_time + forecast_secs)

In [83]:
cmd_bands = list(itertools.chain(*[["-b", str(num)] for num in bands["2t"]]))
result = subprocess.run(
    ["gdal_translate", *cmd_bands, "-of", "NetCDF", "2026-04-24.grib2", "2t.nc"],
    capture_output=True,
    text=True,
)
result.returncode

0

In [91]:
ds = xr.open_dataset("2t.nc")
ds = ds.drop_vars("crs").to_array().to_dataset(name="test")
ds = ds.rename({"variable": "timestamp", "test": "variable"})
ds.coords["timestamp"] = timestamps["2t"]
ds

<xarray.Dataset> Size: 399MB
Dimensions:    (lat: 721, lon: 1440, timestamp: 48)
Coordinates:
  * lat        (lat) float64 6kB -90.0 -89.75 -89.5 -89.25 ... 89.5 89.75 90.0
  * lon        (lon) float64 12kB -180.0 -179.8 -179.5 ... 179.2 179.5 179.8
  * timestamp  (timestamp) int64 384B 1776988800 1776999600 ... 1777496400
Data variables:
    variable   (timestamp, lat, lon) float64 399MB -45.86 -45.86 ... -14.37